In [0]:
from pyspark.sql import functions
from pyspark.sql.types import *

print("BANKING FRAUD MONITORING")
print("CARD TRANSACTIONS AUTO LOADER")

In [0]:
spark.sql("USE CATALOG Capstone_team01")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS bronze
""")

spark.sql("USE SCHEMA bronze")

print("Catalog: Capstone_team01")
print("Schema: bronze")

In [0]:

transaction_path = "/Volumes/capstone_team01/bronze/raw_data/datasets/transaction_data/card_transactions/"

metadata_path = "/Volumes/capstone_team01/bronze/raw_data/autoloader_metadata/card_transactions/"

schema_location = metadata_path + "schema/"

checkpoint_location = metadata_path + "checkpoint/"

print("Transaction source and metadata paths configured")

In [0]:
transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("merchant_id", StringType(), True),
    StructField("card_id", StringType(), True),
    StructField("transaction_ts", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("country", StringType(), True),
    StructField("authorization_code", StringType(), True),
    StructField("payment_status", StringType(), True),
    StructField("device_id", StringType(), True)
])

In [0]:
transaction_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("rescuedDataColumn", "_rescued_data")
    .schema(transaction_schema)
    .load(transaction_path)
)

print("Transaction Auto Loader stream created")

In [0]:
query = (
    transaction_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_location)
    .trigger(availableNow=True)
    .toTable("Capstone_team01.bronze.card_transactions")
)

query.awaitTermination()

print("Card transactions ingested successfully")

In [0]:
bronze_transaction_df = spark.table(
    "Capstone_team01.bronze.card_transactions"
)

display(bronze_transaction_df.limit(10))

In [0]:
bronze_transaction_df.printSchema()

In [0]:
print(
    "Total Bronze transaction records:",
    bronze_transaction_df.count()
)

In [0]:
print("========================================")
print("TRANSACTION AUTO LOADER COMPLETED")
print("========================================")

print("Target: Capstone_team01.bronze.card_transactions")